**Task 1 — Import Libraries and Load Dataset**

In [ ]:
import os
import copy
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.pipeline import make_pipeline

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Libraries imported successfully.")

In [ ]:
df = pd.read_csv("cleaned_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

**Task 2 — Recreate Features and Classification Target**

In [ ]:
# Remove patient_id, cholesterol, and the classification target from the features to stay consistent with Part 2.
y_clf = df["heart_attack_risk"].astype(int).copy()

X = df.drop(
    columns=[
        "patient_id",
        "cholesterol",
        "heart_attack_risk"
    ],
    errors="ignore"
).copy()

print("Original feature shape:", X.shape)
print("Target distribution:")
print(y_clf.value_counts())

text_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in text_columns:
    X[col] = (
        X[col]
        .astype("string")
        .str.strip()
    )

    physical_activity_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

if "physical_activity" in X.columns:

    if X["physical_activity"].isnull().any():
        activity_mode = X[
            "physical_activity"
        ].mode(dropna=True).iloc[0]

        X["physical_activity"] = (
            X["physical_activity"]
            .fillna(activity_mode)
        )

    X["physical_activity"] = (
        X["physical_activity"]
        .map(physical_activity_mapping)
        .astype(int)
    )

print("Physical activity encoding completed.")

# One-hot encode

nominal_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

X_encoded = pd.get_dummies(
    X,
    columns=nominal_columns,
    drop_first=True,
    dtype=int
)

print("Encoded feature shape:", X_encoded.shape)
print("Remaining non-numeric columns:")
print(
    X_encoded
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)



**Task 3 — Train-Test Split and Scaling**

In [ ]:
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print("Scaling completed.")



**Task 4 — Unconstrained Decision Tree Baseline**

In [ ]:
decision_tree_default = DecisionTreeClassifier(
    random_state=42
)

decision_tree_default.fit(
    X_train_scaled,
    y_clf_train
)

default_train_predictions = (
    decision_tree_default.predict(
        X_train_scaled
    )
)

default_test_predictions = (
    decision_tree_default.predict(
        X_test_scaled
    )
)

default_tree_train_accuracy = accuracy_score(
    y_clf_train,
    default_train_predictions
)

default_tree_test_accuracy = accuracy_score(
    y_clf_test,
    default_test_predictions
)

print(
    "Unconstrained Decision Tree training accuracy:",
    round(default_tree_train_accuracy, 4)
)

print(
    "Unconstrained Decision Tree test accuracy:",
    round(default_tree_test_accuracy, 4)
)

print(
    "Train-test accuracy gap:",
    round(
        default_tree_train_accuracy
        - default_tree_test_accuracy,
        4
    )
)

**Task 5 — Controlled Decision Tree**

In [ ]:
# max_depth=5
# min_samples_split=20

decision_tree_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

decision_tree_controlled.fit(
    X_train_scaled,
    y_clf_train
)

controlled_train_predictions = (
    decision_tree_controlled.predict(
        X_train_scaled
    )
)

controlled_test_predictions = (
    decision_tree_controlled.predict(
        X_test_scaled
    )
)

controlled_tree_train_accuracy = accuracy_score(
    y_clf_train,
    controlled_train_predictions
)

controlled_tree_test_accuracy = accuracy_score(
    y_clf_test,
    controlled_test_predictions
)

print(
    "Controlled Decision Tree training accuracy:",
    round(controlled_tree_train_accuracy, 4)
)

print(
    "Controlled Decision Tree test accuracy:",
    round(controlled_tree_test_accuracy, 4)
)

print(
    "Train-test accuracy gap:",
    round(
        controlled_tree_train_accuracy
        - controlled_tree_test_accuracy,
        4
    )
)

# comparison table
tree_comparison = pd.DataFrame({
    "Model": [
        "Unconstrained Decision Tree",
        "Controlled Decision Tree"
    ],
    "Training Accuracy": [
        default_tree_train_accuracy,
        controlled_tree_train_accuracy
    ],
    "Test Accuracy": [
        default_tree_test_accuracy,
        controlled_tree_test_accuracy
    ]
})

tree_comparison["Train-Test Gap"] = (
    tree_comparison["Training Accuracy"]
    - tree_comparison["Test Accuracy"]
)

display(tree_comparison)

tree_comparison.to_csv(
    "results/decision_tree_comparison.csv",
    index=False
)

**Task 6 — Gini vs Entropy Comparison**

In [ ]:
tree_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

tree_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

tree_gini.fit(
    X_train_scaled,
    y_clf_train
)

tree_entropy.fit(
    X_train_scaled,
    y_clf_train
)

gini_test_accuracy = accuracy_score(
    y_clf_test,
    tree_gini.predict(X_test_scaled)
)

entropy_test_accuracy = accuracy_score(
    y_clf_test,
    tree_entropy.predict(X_test_scaled)
)

gini_entropy_comparison = pd.DataFrame({
    "Criterion": [
        "Gini",
        "Entropy"
    ],
    "Test Accuracy": [
        gini_test_accuracy,
        entropy_test_accuracy
    ]
})

display(gini_entropy_comparison)

gini_entropy_comparison.to_csv(
    "results/gini_entropy_comparison.csv",
    index=False
)

Task 7 — Random Forest
n_estimators=100
max_depth=10
random_state=4 **bold text**2

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train_scaled,
    y_clf_train
)

rf_train_predictions = random_forest.predict(
    X_train_scaled
)

rf_test_predictions = random_forest.predict(
    X_test_scaled
)

rf_test_probabilities = (
    random_forest.predict_proba(
        X_test_scaled
    )[:, 1]
)

rf_train_accuracy = accuracy_score(
    y_clf_train,
    rf_train_predictions
)

rf_test_accuracy = accuracy_score(
    y_clf_test,
    rf_test_predictions
)

rf_test_auc = roc_auc_score(
    y_clf_test,
    rf_test_probabilities
)

print(
    "Random Forest training accuracy:",
    round(rf_train_accuracy, 4)
)

print(
    "Random Forest test accuracy:",
    round(rf_test_accuracy, 4)
)

print(
    "Random Forest test AUC:",
    round(rf_test_auc, 4)
)

**Task 8 — Random Forest Feature Importance**

In [ ]:
rf_feature_importance = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Importance":
        random_forest.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Top five Random Forest features:")
display(rf_feature_importance.head(5))

rf_feature_importance.to_csv(
    "results/random_forest_feature_importance.csv",
    index=False
)
# Create and save a feature-importance plot

top_five_features = rf_feature_importance.head(5)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_five_features,
    x="Importance",
    y="Feature"
)

plt.title(
    "Top Five Random Forest Feature Importances"
)

plt.xlabel("Importance Score")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    "plots/random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

**Task 9 — Gradient Boosting**

In [ ]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting.fit(
    X_train_scaled,
    y_clf_train
)

gb_train_predictions = gradient_boosting.predict(
    X_train_scaled
)

gb_test_predictions = gradient_boosting.predict(
    X_test_scaled
)

gb_test_probabilities = (
    gradient_boosting.predict_proba(
        X_test_scaled
    )[:, 1]
)

gb_train_accuracy = accuracy_score(
    y_clf_train,
    gb_train_predictions
)

gb_test_accuracy = accuracy_score(
    y_clf_test,
    gb_test_predictions
)

gb_test_auc = roc_auc_score(
    y_clf_test,
    gb_test_probabilities
)

print(
    "Gradient Boosting training accuracy:",
    round(gb_train_accuracy, 4)
)

print(
    "Gradient Boosting test accuracy:",
    round(gb_test_accuracy, 4)
)

print(
    "Gradient Boosting test AUC:",
    round(gb_test_auc, 4)
)

**Task 10 — Feature Ablation Study**

In [ ]:
lowest_five_features = (
    rf_feature_importance
    .tail(5)
    .sort_values(
        "Importance",
        ascending=True
    )
)

print("Five lowest-importance features:")
display(lowest_five_features)

features_to_remove = (
    lowest_five_features["Feature"]
    .tolist()
)

print("Features to remove:")
print(features_to_remove)

remaining_features = [
    feature
    for feature in X_encoded.columns
    if feature not in features_to_remove
]

remaining_feature_indices = [
    X_encoded.columns.get_loc(feature)
    for feature in remaining_features
]
# Remove the five features from the scaled matrices

X_train_scaled_reduced = (
    X_train_scaled[
        :,
        remaining_feature_indices
    ]
)

X_test_scaled_reduced = (
    X_test_scaled[
        :,
        remaining_feature_indices
    ]
)

print(
    "Full feature count:",
    X_train_scaled.shape[1]
)

print(
    "Reduced feature count:",
    X_train_scaled_reduced.shape[1]
)
# Train the reduced Random Forest

random_forest_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

random_forest_reduced.fit(
    X_train_scaled_reduced,
    y_clf_train
)

rf_reduced_probabilities = (
    random_forest_reduced.predict_proba(
        X_test_scaled_reduced
    )[:, 1]
)

rf_reduced_auc = roc_auc_score(
    y_clf_test,
    rf_reduced_probabilities
)

ablation_comparison = pd.DataFrame({
    "Model": [
        "Full Random Forest",
        "Reduced Random Forest"
    ],
    "Number of Features": [
        X_train_scaled.shape[1],
        X_train_scaled_reduced.shape[1]
    ],
    "Test AUC": [
        rf_test_auc,
        rf_reduced_auc
    ]
})

display(ablation_comparison)

print(
    "AUC change after removing features:",
    round(
        rf_reduced_auc - rf_test_auc,
        6
    )
)

ablation_comparison.to_csv(
    "results/feature_ablation_comparison.csv",
    index=False
)

**Task 11 — Cross-Validated Model Comparison**

In [ ]:
logistic_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

cv_controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

cv_random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

cv_gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
# Create Stratified K-Fold:

stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
# cross-validation

models_for_cv = {
    "Logistic Regression": logistic_model,
    "Controlled Decision Tree":
        cv_controlled_tree,
    "Random Forest": cv_random_forest,
    "Gradient Boosting":
        cv_gradient_boosting
}

cv_results = []

for model_name, model in models_for_cv.items():

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_clf_train,
        cv=stratified_cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV Mean AUC": scores.mean(),
        "CV Std AUC": scores.std()
    })

cv_comparison = pd.DataFrame(
    cv_results
).sort_values(
    "CV Mean AUC",
    ascending=False
).reset_index(drop=True)

display(cv_comparison)

cv_comparison.to_csv(
    "results/cross_validation_comparison.csv",
    index=False
)

**Task 12 — GridSearchCV Pipeline**

The pipeline contains:

Median imputation
Standard scaling
Random Forest

In [ ]:
rf_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
)
# Define the exact required parameter grid:
param_grid = {
    "randomforestclassifier__n_estimators":
        [50, 100, 200],

    "randomforestclassifier__max_depth":
        [5, 10, None],

    "randomforestclassifier__min_samples_leaf":
        [1, 5]
}
number_of_parameter_combinations = (
    3 * 3 * 2
)

number_of_model_fits = (
    number_of_parameter_combinations * 5
)

print(
    "Parameter combinations:",
    number_of_parameter_combinations
)

print(
    "Total model fits across five folds:",
    number_of_model_fits
)

# Run GridSearchCV on the unscaled training data:

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=stratified_cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(
    X_train,
    y_clf_train
)

print("Best parameters:")
print(grid_search.best_params_)

print(
    "Best cross-validation AUC:",
    round(grid_search.best_score_, 6)
)
grid_search_results = pd.DataFrame(
    grid_search.cv_results_
)

selected_grid_columns = [
    "params",
    "mean_train_score",
    "std_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

grid_search_results[
    selected_grid_columns
].sort_values(
    "rank_test_score"
).to_csv(
    "results/grid_search_results.csv",
    index=False
)
# Store the best pipeline:

best_pipeline = grid_search.best_estimator_

best_pipeline_test_probabilities = (
    best_pipeline.predict_proba(
        X_test
    )[:, 1]
)

best_pipeline_test_auc = roc_auc_score(
    y_clf_test,
    best_pipeline_test_probabilities
)

print(
    "Best pipeline test AUC:",
    round(best_pipeline_test_auc, 6)
)


**Task 13 — Manual Learning Curve**

In [ ]:
training_fractions = [
    0.20,
    0.40,
    0.60,
    0.80,
    1.00
]

learning_curve_results = []
training_indices = np.arange(
    len(X_train)
)

rng = np.random.RandomState(42)

rng.shuffle(training_indices)

X_train_shuffled = (
    X_train.iloc[
        training_indices
    ].reset_index(drop=True)
)

y_train_shuffled = (
    y_clf_train.iloc[
        training_indices
    ].reset_index(drop=True)
)
for fraction in training_fractions:

    subset_size = int(
        fraction * len(X_train_shuffled)
    )

    X_subset = X_train_shuffled.iloc[
        :subset_size
    ]

    y_subset = y_train_shuffled.iloc[
        :subset_size
    ]

    subset_pipeline = copy.deepcopy(
        best_pipeline
    )

    subset_pipeline.fit(
        X_subset,
        y_subset
    )

    training_probabilities = (
        subset_pipeline.predict_proba(
            X_subset
        )[:, 1]
    )

    test_probabilities = (
        subset_pipeline.predict_proba(
            X_test
        )[:, 1]
    )

    training_auc = roc_auc_score(
        y_subset,
        training_probabilities
    )

    test_auc = roc_auc_score(
        y_clf_test,
        test_probabilities
    )

    learning_curve_results.append({
        "Training Fraction": fraction,
        "Training Rows": subset_size,
        "Training AUC": training_auc,
        "Test AUC": test_auc
    })

learning_curve_table = pd.DataFrame(
    learning_curve_results
)

display(learning_curve_table)

learning_curve_table.to_csv(
    "results/manual_learning_curve.csv",
    index=False
)
plt.figure(figsize=(9, 6))

plt.plot(
    learning_curve_table[
        "Training Fraction"
    ],
    learning_curve_table[
        "Training AUC"
    ],
    marker="o",
    label="Training AUC"
)

plt.plot(
    learning_curve_table[
        "Training Fraction"
    ],
    learning_curve_table[
        "Test AUC"
    ],
    marker="o",
    label="Test AUC"
)

plt.xlabel("Training Fraction")
plt.ylabel("ROC-AUC")
plt.title("Manual Learning Curve")
plt.legend()

plt.tight_layout()

plt.savefig(
    "plots/manual_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

**Task 14 — Serialize the Best Model**

In [ ]:
joblib.dump(
    best_pipeline,
    "best_model.pkl"
)

print(
    "best_model.pkl saved successfully."
)
model_size_mb = (
    os.path.getsize("best_model.pkl")
    / (1024 ** 2)
)

print(
    "Model size:",
    round(model_size_mb, 2),
    "MB"
)

**Task 15 — Reload and Predict on Two Hand-Crafted Rows**

In [ ]:
# Create two hand-crafted examples from valid feature rows
handcrafted_rows = X_test.iloc[:2].copy()

# Modify selected numeric values
handcrafted_rows.iloc[0, handcrafted_rows.columns.get_loc("age")] = 45
handcrafted_rows.iloc[1, handcrafted_rows.columns.get_loc("age")] = 70

handcrafted_rows.iloc[0, handcrafted_rows.columns.get_loc("bmi")] = 22.0
handcrafted_rows.iloc[1, handcrafted_rows.columns.get_loc("bmi")] = 32.0

handcrafted_rows.iloc[0, handcrafted_rows.columns.get_loc("resting_bp")] = 110
handcrafted_rows.iloc[1, handcrafted_rows.columns.get_loc("resting_bp")] = 155

print("Hand-crafted test rows:")
display(handcrafted_rows)
loaded_model = joblib.load(
    "best_model.pkl"
)

reloaded_predictions = loaded_model.predict(
    handcrafted_rows
)

reloaded_probabilities = (
    loaded_model.predict_proba(
        handcrafted_rows
    )[:, 1]
)

prediction_results = pd.DataFrame({
    "Row": [
        "Hand-crafted Patient 1",
        "Hand-crafted Patient 2"
    ],
    "Predicted Class":
        reloaded_predictions,
    "Positive-Class Probability":
        reloaded_probabilities
})

display(prediction_results)

print(
    "Saved model reloaded and predicted successfully."
)

**Task 16 — Final Model Comparison Table**

In [ ]:
logistic_model.fit(
    X_train_scaled,
    y_clf_train
)

logistic_test_probabilities = (
    logistic_model.predict_proba(
        X_test_scaled
    )[:, 1]
)

logistic_test_auc = roc_auc_score(
    y_clf_test,
    logistic_test_probabilities
)
controlled_tree_test_probabilities = (
    decision_tree_controlled.predict_proba(
        X_test_scaled
    )[:, 1]
)

controlled_tree_test_auc = roc_auc_score(
    y_clf_test,
    controlled_tree_test_probabilities
)
test_auc_mapping = {
    "Logistic Regression":
        logistic_test_auc,

    "Controlled Decision Tree":
        controlled_tree_test_auc,

    "Random Forest":
        rf_test_auc,

    "Gradient Boosting":
        gb_test_auc
}

summary_comparison = (
    cv_comparison.copy()
)

summary_comparison[
    "Test AUC"
] = summary_comparison[
    "Model"
].map(test_auc_mapping)

display(summary_comparison)

summary_comparison.to_csv(
    "results/final_model_comparison.csv",
    index=False
)
tuned_row = pd.DataFrame({
    "Model": [
        "Tuned Random Forest Pipeline"
    ],
    "CV Mean AUC": [
        grid_search.best_score_
    ],
    "CV Std AUC": [
        grid_search_results.loc[
            grid_search.best_index_,
            "std_test_score"
        ]
    ],
    "Test AUC": [
        best_pipeline_test_auc
    ]
})

final_model_comparison = pd.concat(
    [
        summary_comparison,
        tuned_row
    ],
    ignore_index=True
).sort_values(
    "CV Mean AUC",
    ascending=False
).reset_index(drop=True)

display(final_model_comparison)

final_model_comparison.to_csv(
    "results/final_model_comparison_with_tuned.csv",
    index=False
)
# Identify the recommended model:
recommended_model = final_model_comparison.iloc[0]

print(
    "Recommended model:",
    recommended_model["Model"]
)

print(
    "CV mean AUC:",
    round(
        recommended_model["CV Mean AUC"],
        4
    )
)

print(
    "Test AUC:",
    round(
        recommended_model["Test AUC"],
        4
    )
)

**Task 17 — Save a Complete Results Summary**

In [ ]:
part3_summary = pd.DataFrame({
    "Metric": [
        "Default Tree Train Accuracy",
        "Default Tree Test Accuracy",
        "Controlled Tree Train Accuracy",
        "Controlled Tree Test Accuracy",
        "Gini Tree Test Accuracy",
        "Entropy Tree Test Accuracy",
        "Random Forest Train Accuracy",
        "Random Forest Test Accuracy",
        "Random Forest Test AUC",
        "Reduced Random Forest Test AUC",
        "Gradient Boosting Train Accuracy",
        "Gradient Boosting Test Accuracy",
        "Gradient Boosting Test AUC",
        "Grid Search Best CV AUC",
        "Best Pipeline Test AUC"
    ],
    "Value": [
        default_tree_train_accuracy,
        default_tree_test_accuracy,
        controlled_tree_train_accuracy,
        controlled_tree_test_accuracy,
        gini_test_accuracy,
        entropy_test_accuracy,
        rf_train_accuracy,
        rf_test_accuracy,
        rf_test_auc,
        rf_reduced_auc,
        gb_train_accuracy,
        gb_test_accuracy,
        gb_test_auc,
        grid_search.best_score_,
        best_pipeline_test_auc
    ]
})

display(part3_summary)

part3_summary.to_csv(
    "results/part3_summary.csv",
    index=False
)

**Task 18 — Create and Download requirements.txt**

In [ ]:
requirements = """pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
"""

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(requirements)

print(
    "requirements.txt created successfully."
)
files.download(
    "requirements.txt"
)

**Task 19 — Download Generated Files**

In [ ]:
files.download(
    "best_model.pkl"
)
import shutil

shutil.make_archive(
    "part3_plots",
    "zip",
    "plots"
)

files.download(
    "part3_plots.zip"
)
shutil.make_archive(
    "part3_results",
    "zip",
    "results"
)

files.download(
    "part3_results.zip"
)